# 第8周 · PyTorch 与训练循环（Capstone 启动）

**本周结束时你能做到**：创建与操作 tensor（shape、dtype、device）并解释"autograd 就是张量版的 micrograd"；用 `nn.Module` 定义 MLP；凭记忆写出训练循环的五步核心；用 `Dataset`/`DataLoader` 组织数据，在 MNIST 上跑通带验证集的完整训练；把 MLP 与逻辑回归、随机森林放在同一划分上公平比较。本周还将启动 Capstone：学会用 bit flip 方法把"领域默认假设"变成研究问题。

本周蓝本是 PyTorch 官方教程 [Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)（官方示例用 FashionMNIST，我们课上用 MNIST——下周轮到 FashionMNIST，你等于把官方教程的数据也见过一遍）。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from zero2researcher import data, experiments
from zero2researcher.checks import week08 as ck

# 课程统一的 device 选择：有 CUDA 用 CUDA（Colab GPU），Apple 芯片用 MPS，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

---
## S1 · tensor 与 autograd：张量版的 micrograd

上周你从零写出了 micrograd 的 `Value`：一个数，带着它的梯度和反向传播规则。PyTorch 的核心对象 **tensor** 做的是同一件事，只是把"一个数"换成"一个多维数组"，并把所有运算搬进了高度优化的 C++/GPU 内核。逐项对应：

| 第 7 周的 micrograd | PyTorch |
| --- | --- |
| `Value(2.0)` | `torch.tensor(2.0, requires_grad=True)` |
| `v.data` | tensor 本身存的数值 |
| `v.grad` | `t.grad` |
| `loss.backward()` | `loss.backward()`（连名字都没变） |
| 你为每个运算手写的 `_backward` 闭包 | autograd 引擎自动生成 |
| 标量，一次算一个数 | 张量，一次算一批 |

先熟悉 tensor 本身——创建、shape、dtype：

In [ ]:
# ✍️ 跟着老师一起写
pass
print(a)
print("shape:", a.shape, "| dtype:", a.dtype, "| device:", a.device)
print("b:", tuple(b.shape), "  c:", tuple(c.shape))

tensor 与 NumPy 数组无缝互转，第 2–6 周的全部数组技能（切片、聚合、广播）原样适用：

In [ ]:
arr = np.arange(6, dtype=np.float32).reshape(2, 3)
t = torch.from_numpy(arr)
print(t)
print("切片 t[0]:", t[0].tolist(), "| 聚合 t.mean():", t.mean().item(), "| 转回 NumPy:", type(t.numpy()).__name__)

### autograd：同一个表达式，第三种解法

第 6 周你对 $f(x) = (3x + 2)^2$ 手推过链式法则（答案 30），第 7 周 micrograd 把它自动化。现在换 PyTorch：给 tensor 挂上 `requires_grad=True`，正常算前向，然后 `backward()`：

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
# ✍️ 跟着老师一起写
pass
print("f =", f.item(), "   df/dx =", x.grad.item())   # 第6周手推 = 第7周 micrograd = 30.0

和 micrograd 一样，`.grad` 是**累加**的——再算一次 backward，它不会被覆盖而是叠上去：

In [ ]:
f2 = (3 * x + 2) ** 2
f2.backward()
print("再 backward 一次:", x.grad.item(), "   ← 变成 60 了：梯度在累加")
x.grad.zero_()
print("手动清零之后:", x.grad.item(), "   ← 训练循环每一步都要做这件事（micrograd 的 zero_grad）")

### 把计算搬上 GPU

tensor 可以整体搬到加速设备上（Colab：代码执行程序 → 更改运行时类型 → T4 GPU）。课程代码统一用开头 imports 里那行"自动选择"——有 CUDA 用 CUDA、Apple 芯片用 MPS、否则 CPU，同一份代码三种机器都能跑：

In [ ]:
big = torch.randn(1000, 1000)
big_dev = big.to(device)                      # 搬上设备；模型同理：model.to(device)
print("数据在哪:", big_dev.device, "| 矩阵乘的结果 shape:", tuple((big_dev @ big_dev).shape))

---
## S2 · nn.Module 与五步训练循环

### 定义模型：照着写

第 3 周我们约定过：类的写法先"照着写"，骨架记熟就行。PyTorch 的所有模型都长一个样——继承 `nn.Module`，**`__init__` 里定义层，`forward` 里定义计算**：

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10):
        super().__init__()
        # ✍️ 跟着老师一起写
        pass

    def forward(self, x):
        # ✍️ 跟着老师一起写
        pass


model = MLP().to(device)
print(model)
print("参数量:", sum(p.numel() for p in model.parameters()))   # 784×256 + 256 + 256×10 + 10 = 203,530

`model.parameters()` 一次性交出所有可训练参数——第 7 周 micrograd 的 `MLP.parameters()` 就是它的标量版。优化器拿到它们就知道该更新谁。

### 损失函数：nn.CrossEntropyLoss 要的是 logits

**本周的头号 bug 来源，先正面讲清楚。** [官方文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)原话："The input is expected to contain the unnormalized logits for each class"。也就是：

- 输入：模型的原始输出 **logits**，形状 `(N, 10)`——**不要先过 softmax**（softmax 已经包含在损失里，它等价于 LogSoftmax + NLLLoss）；
- 标签：类别**索引**（0–9 的整数，long 型），形状 `(N,)`——不是 one-hot。

$$\text{CE}(z, y) = -\log \frac{e^{z_y}}{\sum_j e^{z_j}}$$

把 softmax 后的概率再喂给它**不会报错**——损失照样下降、模型照样"在学"，只是又慢又封顶。这种"静默 bug"是下周排查课的主角之一，先看一眼它长什么样：

In [ ]:
criterion = nn.CrossEntropyLoss()
logits = torch.tensor([[2.0, 0.5, -1.0], [0.1, 3.0, 0.2]])   # 2 个样本、3 类
labels = torch.tensor([0, 1])                                 # 类别索引，不是 one-hot
print("正确用法（logits 直接进）:", round(criterion(logits, labels).item(), 4))
print("静默 bug（先 softmax 再进）:", round(criterion(torch.softmax(logits, dim=1), labels).item(), 4),
      "← 不报错，但数已经不对了")

### 训练循环的五步核心

每个 batch 都做同样的五件事——**本周结束前要背下来**：

```python
logits = model(xb)                 # 1 前向
loss = criterion(logits, yb)       # 2 算损失
optimizer.zero_grad()              # 3 清零梯度（.grad 是累加的！）
loss.backward()                    # 4 反向传播
optimizer.step()                   # 5 更新参数
```

第 7 周你在 micrograd 里手写过完全相同的循环：前向 → 损失 → `zero_grad()` → `backward()` → `p.data -= lr * p.grad`。`torch.optim` 只是把最后一步抽象成可替换的优化器。在一个随机 batch 上走一遍：

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
xb = torch.randn(32, 1, 28, 28, device=device)
yb = torch.randint(0, 10, (32,), device=device)

# ✍️ 跟着老师一起写
pass
w = next(model.parameters())
print("loss =", round(loss.item(), 4), "| backward 之后参数有梯度:", w.grad is not None)

---
## S3 · Dataset、DataLoader 与完整训练

### Dataset 协议：你第 3 周就写过了

PyTorch 的 map-style `Dataset` 必须实现 `__getitem__`（按下标取一个样本）；`DataLoader` 的默认采样器还要用到 `__len__`（数据集有多大）。第 3 周作业的 `TabularDataset` 两个都有，所以 `DataLoader` 可以直接用它。官方文档建议写成 `class TabularDataset(torch.utils.data.Dataset)` 的子类，效果相同。`DataLoader` 负责其余所有脏活：打乱、切 batch、把样本拼成张量：

In [ ]:
from torch.utils.data import DataLoader, Subset


class TabularDataset:                  # 第 3 周作业原样搬来，一个字都不用改
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


toy = TabularDataset(torch.randn(10, 3), torch.arange(10))
loader = DataLoader(toy, batch_size=4, shuffle=True)
for xb_, yb_ in loader:
    print("batch:", tuple(xb_.shape), "标签:", yb_.tolist())

三个术语（之后天天用）：

- **batch**：一次前向/反向用的一小撮样本（上面是 4 个）；
- **iteration**：处理一个 batch = 五步做一遍；
- **epoch**：整个数据集过一遍。10 个样本、batch_size=4 → 一个 epoch 有 3 个 iteration（`len(loader)` 告诉你）。

### MNIST 与两个"魔法数字"

MNIST：60,000 张训练 + 10,000 张测试的 28×28 手写数字（LeCun et al., 1998），torchvision 一行下载。几乎所有教程都写 `Normalize((0.1307,), (0.3081,))`——这两个数从哪来？**自己算一遍就祛魅了**：它们就是全部训练像素的均值和标准差（出处：pytorch/examples 仓库的 mnist/main.py）。

In [ ]:
from torchvision import datasets, transforms

def load_dataset(cls, train, transform=None):
    """下载并缓存数据集（MNIST 约 11MB，只有第一次要等）；网络抖动时自动重试一次。"""
    for attempt in (1, 2):
        try:
            return cls(root=data.download_root(), train=train, download=True, transform=transform)
        except Exception as e:
            if attempt == 1:
                print("下载失败，重试一次：", e)
            else:
                raise


raw = load_dataset(datasets.MNIST, train=True)   # 先不加 transform：.data 是 0–255 的整数张量
# ✍️ 跟着老师一起写
pass
print(f"全部训练像素的 mean = {mean:.4f}, std = {std:.4f}   ← 0.1307 和 0.3081 就是这么来的")

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(),                        # (28,28) 的 0-255 → (1,28,28) 的 0-1
                          transforms.Normalize((0.1307,), (0.3081,))])  # 再标准化到均值 0、方差 1
train_full = load_dataset(datasets.MNIST, train=True, transform=tfm)
test_set = load_dataset(datasets.MNIST, train=False, transform=tfm)

# 课程固定子集：训练用前 10,000 张；验证集取训练集末段不重叠的 1,000 张；测试集全量 10,000 张
train_set = Subset(train_full, range(0, 10000))
val_set = Subset(train_full, range(50000, 51000))
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=512)
test_loader = DataLoader(test_set, batch_size=512)
print(len(train_set), len(val_set), len(test_set), "| 每个 epoch 的 iteration 数:", len(train_loader))

### 带验证集的完整循环

比五步核心多两件事：

- `model.train()` / `model.eval()`：切换训练/评估模式（Dropout、BatchNorm 等层在两种模式下行为不同——下周见真章，今天先养成习惯）；
- `torch.no_grad()`：评估时不记录计算图——省时省内存。

注意两者的分工：`no_grad` **只**关掉梯度记录，并不保证"评估不改模型"。评估不改模型靠的是两件事：**不调用 `optimizer.step()`**，以及 **`model.eval()`**——像 BatchNorm 这样的层，在 train 模式下即使包在 `no_grad` 里，每次前向也会更新它自己记录的均值/方差（running 统计量）。所以评估时 `eval()` 和 `no_grad()` **两个都要用**。一个小演示：

In [ ]:
bn = nn.BatchNorm1d(4)                        # 下周细讲的一种层；这里只看它记录的 running_mean（初始全 0）
with torch.no_grad():                         # 关掉了梯度记录……
    bn(torch.randn(32, 4) + 3.0)              # ……但没有调用 bn.eval()：仍是 train 模式，前向一次
print("no_grad 里前向一次之后的 running_mean:", bn.running_mean.numpy().round(2), "← 不再是全 0，模型被改了")

把它们各写成一个函数（这两个函数你本周作业还要凭记忆再写一遍）：

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch，返回平均训练损失。"""
    # ✍️ 跟着老师一起写
    pass


def evaluate(model, loader, device):
    """返回准确率；eval 模式 + 全程 no_grad，绝不改模型。"""
    # ✍️ 跟着老师一起写
    pass


In [ ]:
experiments.set_seed(42)
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = {"loss": [], "val_acc": []}
for epoch in range(3):
    tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_acc = evaluate(model, val_loader, device)
    history["loss"].append(tr_loss)
    history["val_acc"].append(val_acc)
    print(f"epoch {epoch + 1}: train loss {tr_loss:.4f} | val acc {val_acc:.4f}")

mlp_acc = evaluate(model, test_loader, device)
print("测试准确率:", round(mlp_acc, 4))
ck.mnist_mlp_accuracy(mlp_acc)

In [ ]:
epochs = range(1, len(history["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot(epochs, history["loss"], "o-")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mean training loss")
axes[0].set_title("Training loss falls epoch by epoch")
axes[1].plot(epochs, history["val_acc"], "o-", color="C1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation accuracy")
axes[1].set_title("Validation accuracy passes 0.93 within 3 epochs")
for ax in axes:
    ax.set_xticks(list(epochs))
plt.tight_layout()
plt.show()

---
## S4 · Capstone 启动 + 三方对比

### 什么是好的 Capstone

课程第 15–20 周，你将独立完成一项小研究并写成报告——从今天开始孵化选题。好的 Capstone 不是"做一个很大的模型"，而是**干净地回答一个清楚的问题**。三条选题标准，一条都不能少：

1. **数据可得**：公开数据集，或你实验室里**现在**就拿得到的数据——不能指望"到时候再采"；
2. **Colab 可跑**：免费 GPU 几小时内能完成全部实验（本周和下周你会学会估算训练时间）；
3. **有明确基线**：存在一个"不聪明但诚实"的参照（多数类、逻辑回归、随机森林……）。你的问题永远是"能否超过它、超过多少"，而不是"我的模型准确率是多少"。

### bit flip：把"默认假设"翻转成研究问题

斯坦福 CS197（AI 研究方法课）教的选题方法：研究贡献 = 找到领域里一个大家**不加论证就接受的假设**（bit），把它翻过来（flip），并写出**可检验的预测**。一个完整示例：

> - **默认假设（bit）**：深度学习时代，神经网络做图像分类理所当然优于经典机器学习。
> - **翻转（flip）**：小数据、简单任务上，经典方法（如随机森林）可能不输甚至更强。
> - **可检验预测（prediction）**：MNIST 的 10,000 张训练子集上，随机森林的测试准确率 ≥ 一个未精调的小 MLP；训练数据再缩小 5 倍，差距进一步扩大。

这个示例不是随便挑的：**下面马上检验它的前半句**，第 9 周的正式研究实验检验它的后半句。作业 C 里你将用同样的格式写 3 个自己的候选问题（第 10 周提交一页纸提案）。

### 三方对比：同一划分上的 MLP、逻辑回归、随机森林

公平比较的底线（第 5、6 周反复强调的）：**同一训练集、同一测试集**。三位选手都用刚才那 10,000 张训练图，逻辑回归和随机森林把图展平成 784 维向量：

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_train = train_full.data[:10000].reshape(10000, -1).numpy() / 255.0   # 与 MLP 完全相同的 1 万张
y_train = train_full.targets[:10000].numpy()
X_test = test_set.data.reshape(-1, 784).numpy() / 255.0
y_test = test_set.targets.numpy()

# ✍️ 跟着老师一起写
pass
print(f"逻辑回归 {lr_acc:.4f} | 随机森林 {rf_acc:.4f} | MLP {mlp_acc:.4f}")
ck.mnist_lr_accuracy(lr_acc)
ck.mnist_rf_accuracy(rf_acc)

In [ ]:
names = ["Logistic regression", "Random forest", "MLP (3 epochs)"]
vals = [lr_acc, rf_acc, mlp_acc]
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(names, vals, color=["C0", "C1", "C2"])
ax.bar_label(bars, fmt="%.3f")
ax.set_ylim(0.85, 1.0)
ax.set_ylabel("Test accuracy")
ax.set_title("On 10k MNIST images the random forest edges out the MLP")
plt.show()

随机森林略胜 MLP——bit flip 示例的前半句成立。三个要点：

- **深度学习在小数据、简单任务上未必赢**。它的优势要在数据量大、结构复杂（图像语义、文本语序）时才兑现——第 10 周起你会亲眼看到它反超；
- 这不是"深度学习不行"，而是"没有免费午餐"：**每个结论都带着数据量与任务的限定条件**，去掉限定条件的结论都是标题党；
- "小样本下结论会不会反转"正是第 9 周正式研究实验的主角——而且那次 MLP 会被允许调参。

### 本周小结

- tensor = 带 autograd 的多维数组；`.grad` 累加，所以每步都要清零
- `nn.Module`：`__init__` 定义层、`forward` 定义计算；`nn.CrossEntropyLoss` 吃 **logits + 类别索引**（先 softmax 是静默 bug）
- 五步核心：**前向 → 损失 → zero_grad → backward → step**（官方教程把清零放循环末尾，等价）
- Dataset：必须有 `__getitem__`，`DataLoader` 默认还要 `__len__`（第 3 周写过）；DataLoader 管打乱与切 batch；epoch / iteration / batch
- 评估三件套：`model.eval()` + `torch.no_grad()` + 绝不更新参数（`no_grad` 只管梯度记录；BatchNorm 之类的层会不会改自己，由 `train()` / `eval()` 决定）
- 小数据简单任务上随机森林不输 MLP——结论都带条件
- bit flip：默认假设 → 翻转 → 可检验预测；三条选题标准：数据可得、Colab 可跑、有明确基线